# Sample Transaction DataFrame

A simple Spark DataFrame created from sample transaction data with date, transaction_type, and amount columns.

In [0]:
data = [
    ("01/09/2026", "credite", 20),
    ("02/09/2026", "debit", 40),
]

columns = ["date", "transaction_type", "amount"]

df = spark.createDataFrame(data, columns)
display(df)

In [0]:

from pyspark.sql.functions import *

In [0]:
#reust i want like this sep-2026 -20
df1 = df.withColumn("date", date_format(to_date(col("date"), "dd/MM/yyyy"), "MMM-yyyy"))
df2 = df1.withColumn("new_amount", when(col("transaction_type")== "debit", -col("amount")).otherwise(col("amount")))
#display(df2)

final_df = df2.groupBy("date").agg(sum("new_amount").alias("total_amount"))

display(final_df)

In [0]:
# impelementing word count logic
# Data
data = [ ("hello spark"), ("hello hadoop"), ("spark pyspark"),("spark hi")]
# columns
columns = ["values"]
# creatinf df
df = spark.createDataFrame(data,columns)
df.display()


# now will write the logic
df1 = (df.select(explode(split(col("values"), " ")).alias("word"))
       .groupBy(col("word"))
       .agg(count("*").alias("word_count"))
       .orderBy(col("word_count").desc())
    )
display(df1)
df1.show()

"""
SELECT word, COUNT(*) AS count
FROM (
    SELECT explode(split(value, ' ')) AS word FROM text_table
)
GROUP BY word
"""

In [0]:
# DataFrame 1 (larger) - employees
data1 = [
    (1, "Alice", "Engineering"),
    (2, "Bob", "Sales"),
    (3, "Charlie", "Engineering"),
    (4, "Diana", "Marketing"),
    (5, "Eve", "Sales"),
]
cols1 = ["emp_id", "name", "dept"]
df1 = spark.createDataFrame(data1, cols1)

# DataFrame 2 (smaller) - department info per employee (good candidate for broadcast)
data2 = [
    (1, "San Francisco", 50000),
    (2, "New York", 40000),
    (3, "San Francisco", 50000),
    (4, "Los Angeles", 45000),
    (5, "New York", 40000),
]
cols2 = ["emp_id", "location", "budget"]
df2 = spark.createDataFrame(data2, cols2)

# Broadcast join: small df2 is broadcast to all executors to avoid shuffle
joined_df = df1.join(broadcast(df2), "emp_id", "inner")

display(joined_df)

In [0]:
# Q Rename Column (name → first_name)
emp_data = [
(1, "Vinay", 50000),
(2, "Raj", 60000),
(3, "Sam", 70000)
]
emp_cols = ["emp_id", "name", "sal"]
emp_df = spark.createDataFrame(emp_data, emp_cols)
emp_df.display()

new_df = emp_df.withColumnRenamed("name", "first_name")
new_df.display()

new_df2 = emp_df.toDF(
"new_emp_id",
"first_name",
"salary"
).show()



# some additional points on the same questions.
"""1. Using withColumnRenamed() (Best Way)
Python
new_df = emp_df.withColumnRenamed("name", "first_name")

✅ Most commonly used
 ✅ Easy to read
 ✅ Interview-friendly

2. Using select() and Alias
Python
from pyspark.sql.functions import col
 
new_df = emp_df.select(
"emp_id",
col("name").alias("first_name"),
"dep_id",
"sal"
)

✅ Useful when selecting only required columns

3. Using toDF()

If you want to rename all columns at once:

Python
new_df = emp_df.toDF(
"emp_id",
"first_name",
"dep_id",
"sal"
)

⚠️ All column names must be supplied in order."""

In [0]:
from pyspark.sql.functions import *
# Q Rename all Columns (with prefix new_ assume u have 1000 columns)
emp_data = [
(1, "Vinay", 50000),
(2, "Raj", 60000),
(3, "Sam", 70000)
]
emp_cols = ["emp_id", "name", "sal"]
emp_df = spark.createDataFrame(emp_data, emp_cols)
print("Before")
emp_df.display()
new_df = emp_df.toDF(*[f"new_{c}" for c in emp_df.columns])
print("After")
display(new_df)

some additionals points:

What is List Comprehension?

List comprehension is a shorter way of writing a for loop to create a list.
Normal for Loop
Suppose you want a list of numbers:
Python
nums = []
for i in range(5):
nums.append(i)
print(nums)
Output:
Python
[0, 1, 2, 3, 4]

Using List Comprehension
Python
nums = [i for i in range(5)]
print(nums)
"""

Using Normal Loop
Python
new_cols = []
for c in df.columns:
new_cols.append("new_" + c)
print(new_cols)

Using List Comprehension
Python
new_cols = [f"new_{c}" for c in df.columns]
print(new_cols)

Output:
Python
['new_emp_id', 'new_name', 'new_sal']

For every column c in df.columns, create a new value "new_" + c
and store it in a list.

Easy Interview Definition: what is * here??
* is the Python unpacking operator. It takes elements from a list, tuple, or iterable and passes them as individual arguments to a function.

Example:
Python
cols = ['new_emp_id', 'new_name', 'new_sal']
df.toDF(*cols)

Here *cols unpacks the list into separate column-name arguments for toDF().

q) to add pre fix and suff-fix to all the columns 

new_cols = [f"new_{c}" for c in df.columns]
print(new_cols)

Output:
Python
pre fix:
new_df = df.toDF(*[f"_new{c}" for c in df.columns])
Suffixs:
new_df = df.toDF(*[f"{c}_new" for c in df.columns])

In [0]:
##3. Second Highest Salary in Each Department
emp_data = [
(1, "Vinay", 10, 90000),
(2, "Raj", 10, 80000),
(3, "Sam", 10, 70000),
(4, "John", 20, 95000),
(5, "Mary", 20, 85000),
(6, "David", 20, 75000),
(7, "Ravi", 30, 60000),
(8, "Tom", 30, 50000)
]

emp_cols = ["emp_id", "name", "dep_id", "sal"]

emp_df = spark.createDataFrame(emp_data, emp_cols)


dep_data = [
(10, "IT"),
(20, "HR"),
(30, "Finance")
]

dep_cols = ["dep_id", "dep_name"]

dep_df = spark.createDataFrame(dep_data, dep_cols)


# solution am using window funcation with dense_rank()
from pyspark.sql.window import Window
from pyspark.sql.functions import *
# joining the 2 dfs 
df_join = emp_df.join(dep_df, "dep_id", "inner")
window_spec = Window.partitionBy("dep_name").orderBy(col("sal").desc())

dep_sal_df = df_join.withColumn("rnk", dense_rank().over(window_spec))

#result
final_df = dep_sal_df.filter(col("rnk") == 2).show()

In [0]:
# Departments With No Employees 
emp_data = [
(1, "Vinay", 10, 50000),
(2, "Raj", 20, 60000),
(3, "Sam", 10, 70000)
]

emp_cols = ["emp_id", "name", "dep_id", "sal"]

emp_df = spark.createDataFrame(emp_data, emp_cols)


dep_data = [
(10, "IT"),
(20, "HR"),
(30, "Finance"),
(40, "Admin"),
(50, "Legal")
]

dep_cols = ["dep_id", "dep_name"]

dep_df = spark.createDataFrame(dep_data, dep_cols)
from pyspark.sql.functions import *
# Joing using riht join
df = dep_df.join(emp_df, "dep_id", "left_anti").show()
# Departments with no employees


In [0]:
# Join With NULL Practice
emp_data = [
(1, "Vinay", 10),
(2, "Raj", None),
(3, "Sam", 10),
(4, "John", None)
]

emp_cols = ["emp_id", "name", "dep_id"]

emp_df = spark.createDataFrame(emp_data, emp_cols)


dep_data = [
(10, "IT"),
(None, "HR"),
(30, "Finance")
]

dep_cols = ["dep_id", "dep_name"]

dep_df = spark.createDataFrame(dep_data, dep_cols)

join_df = emp_df.join(dep_df, "dep_id", "full_outer")
join_df.show()

In [0]:
print("inner")
emp_df.join(dep_df, "dep_id", "inner").show()
print("left")
emp_df.join(dep_df, "dep_id", "left").show()
print("right")
emp_df.join(dep_df, "dep_id", "right").show()
print("full or fullouter")
emp_df.join(dep_df, "dep_id", "full_outer").show()
print("left_semi")
emp_df.join(dep_df, "dep_id", "left_semi").show()
print("left_anti")
emp_df.join(dep_df, "dep_id", "left_anti").show()

In [0]:
# dataset for the leading zeros join question.
# Join IDs Like 0001 and 1 Without Casting or Substring
data1 = [
("0001", "Vinay"),
("0002", "Raj"),
("0003", "Sam"),
("0010", "John"),
("0100", "David")
]
cols1 = ["id", "name"]
df1 = spark.createDataFrame(data1, cols1)
df1.show()

data2 = [
("1", "IT"),
("2", "HR"),
("3", "Finance"),
("10", "Admin"),
("100", "Legal")
]
cols2 = ["id", "department"]
df2 = spark.createDataFrame(data2, cols2)
df2.show()

##
from pyspark.sql.functions import *
df3 = (
    df1.join(
        df2,
        regexp_replace(df1.id, "^0+", "") ==
        regexp_replace(df2.id, "^0+", ""),
        "inner"
        )
).show()


"""
Syntax
Python
regexp_replace(column, pattern, replacement)
column → column to modify
pattern → regex pattern to search
replacement → value to replace with
Understanding ^0+
^

Means:

Plain Text
Start of the string

Examples:

Plain Text
^A -> A must be first character
^0 -> 0 must be first character
0

Matches the digit:

Plain Text
0
+

Means:
Plain Text
One or more occurrences

Examples:
Plain Text
0+ matches
0
00
000
0000
Combined: ^0+
Plain Text
Start of string
+
One or more zeros

Matches:

Plain Text
0001 -> matches 000
0012 -> matches 00
01 -> matches 0
10 -> no match
Replacement
Python
regexp_replace(col("id"), "^0+", "")
``

Replace the matched leading zeros with:

Plain Text
""

(empty string)

Examples
Plain Text
Input Output

0001 -> 1
0012 -> 12
01 -> 1
00010 -> 10
10 -> 10
123 -> 123
Visual Representation

For:

Plain Text
000123

Regex:

Plain Text
^0+
↑↑↑

matches:

Plain Text
000

Replace with:

Plain Text
""

Result:

Plain Text
123
Join Example
Table A
Plain Text
id
----
0001
0002
0003
Table B
Plain Text
id
----
1
2
3

Direct join:

Python
df1.join(df2, "id")

No match because:

Plain Text
0001 != 1
"""

# Commonly Used Built-in String Functions
upper()
lower()
trim()
ltrim()
rtrim()
length()
substring()
concat()
split()
regexp_replace()
regexp_extract()

In [0]:
""" 
Spark Questions - Common String Functions
1. upper()
Converts all characters to uppercase.
Syntax
upper(column)
column → column to convert to uppercase
Example
upper(col("name"))

Input:
vinay

Output:
VINAY

2. lower()
Converts all characters to lowercase.
Syntax
lower(column)
column → column to convert to lowercase
Example
lower(col("name"))

Input:
VINAY

Output:
vinay

3. trim()
Removes spaces from both the beginning and end.

Syntax
trim(column)

column → column whose spaces need to be removed
Example
trim(col("name"))


Input:
"  Vinay  "

Output:
"Vinay"

4. ltrim()
Removes spaces from the left side only.

Syntax
ltrim(column)

column → column whose left-side spaces are removed
Example
ltrim(col("name"))


Input:
"  Vinay"

Output:
"Vinay"

5. rtrim()
Removes spaces from the right side only.
Syntax
rtrim(column)
column → column whose right-side spaces are removed
Example
rtrim(col("name"))

Input:
"Vinay   "

Output:
"Vinay"

6. length()

Returns the number of characters in a string.
Syntax
length(column)
column → column whose length is calculated
Example
length(col("name"))


Input:
Vinay

Output:
5

7. substring()
Extracts part of a string.
Syntax
substring(column, start_position, length)

column → source column
start_position → starting position (starts from 1)
length → number of characters to extract
Example
substring(col("name"), 1, 3)


Input:
Vinay

Output:
Vin

8. concat()

Combines multiple columns into one string.

Syntax
concat(column1, column2, ...)
column1, column2 → columns to combine
Example
concat(col("fname"), col("lname"))


Input:
Vinay
Kumar


Output:
VinayKumar

9. split()

Splits a string into an array using a delimiter.

Syntax
split(column, delimiter)

column → column to split
delimiter → separator
Example
split(col("name"), " ")


Input:
Vinay Kumar


Output:
["Vinay","Kumar"]

10. regexp_replace()

Replaces matching text using a regular expression.

Syntax
regexp_replace(column, pattern, replacement)

column → column to modify
pattern → regex pattern to search
replacement → value to replace with
Example
regexp_replace(col("id"), "^0+", "")


Input:
000123


Output:
123

11. regexp_extract()
Extracts matching text using a regular expression.

Syntax
regexp_extract(column, pattern, group_number)

column → column to search
pattern → regex pattern
group_number → matching group to return
Example
regexp_extract(col("email"), "(.*)@", 1)


Input:
vinay@gmail.com


Output:
vinay

Interview Quick Revision
upper()


→ Convert to UPPERCASE

lower()


→ Convert to lowercase

trim()


→ Remove spaces from both sides

ltrim()


→ Remove left spaces

rtrim()


→ Remove right spaces

length()


→ Count characters

substring()


→ Extract part of string

concat()


→ Combine strings

split()


→ Convert string to array

regexp_replace()


→ Replace matching text

regexp_extract()


→ Extract matching text from string

These are among the most frequently used string functions in PySpark interviews and real projects. ✅ """

In [0]:
"""
Interview One-Liner

f"" is a formatted string literal in Python. It allows variables or expressions to be embedded directly inside a string using {} placeholders, making string construction more readable than using concatenation (+). ✅

Curly Braces {} in f-strings
Curly braces tell Python:

"Take the value inside the braces and insert it into the string."

Syntax = Python:
f"text {variable} text"
f → formatted string
{variable} → value to insert
"""